# UCS420 Assignment 4
### Cognitive FAQ System using Pandas

Gurkirpa Singh, 3Q34, 1024170378

In [2]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

## Ques 1

My roll no is 1024170378, so the last two digits are 7 and 8. 

In [3]:
# Q1
roll = "1024170378"
digits = [int(c) for c in roll]
cats = ["billing", "account", "general"]

last_two = digits[-2:]
print("roll:", roll)
print("digits:", digits)
for d in last_two:
    print("digit", d, "->", cats[d % 3])

roll: 1024170378
digits: [1, 0, 2, 4, 1, 7, 0, 3, 7, 8]
digit 7 -> account
digit 8 -> general


In [4]:
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

# my two entries, from digit 7 and digit 8
my_entries = [
    {"question": "how do i update my registered mobile number",
     "answer": "Log in and open Profile > Contact Details to change your mobile number.",
     "keywords": "mobile number update", "category": cats[last_two[0] % 3]},
    {"question": "where is the campus help desk located",
     "answer": "The help desk is on the ground floor of the main academic block.",
     "keywords": "help desk location", "category": cats[last_two[1] % 3]},
]

df = pd.DataFrame(fixed_entries + my_entries)
print(df)

                                      question                                                       answer                keywords category
0                       what is the annual fee                                    The annual fee is Rs 500.   fee cost price charge  billing
1                        how to reset password                             Go to Settings > Reset Password.    password reset login  account
2                  what are your working hours                                    We are open 9 AM to 5 PM.  hours timing open time  general
3                        how can i pay the fee                   You can pay via UPI, card, or net banking.     pay payment upi fee  billing
4  how do i update my registered mobile number  Log in and open Profile > Contact Details to change your...    mobile number update  account
5        where is the campus help desk located  The help desk is on the ground floor of the main academi...      help desk location  general


## Ques 2

Scoring is just word overlap. I break the query into words, throw away the very
common ones like "how" and "the" since they match everything, and then check how
many of the remaining words show up in an entry's keywords or question. Score is
the fraction that matched, so 1.0 means every useful word in the query was found.

In [5]:
# Q2
stop = {"how", "what", "the", "is", "a", "an", "to", "do", "i", "my", "can", "are", "your"}


def score(query, df):
    words = set(query.lower().split()) - stop
    if not words:
        return pd.DataFrame()
    rows = []
    for i, row in df.iterrows():
        bag = set(row["keywords"].split()) | set(row["question"].split())
        hits = words & bag
        if hits:   # only keep entries that matched at least one word
            rows.append({
                "question": row["question"],
                "category": row["category"],
                "matched": " ".join(sorted(hits)),
                "confidence": round(len(hits) / len(words), 2),
            })
    res = pd.DataFrame(rows)
    if not res.empty:
        res = res.sort_values("confidence", ascending=False).reset_index(drop=True)
    return res


print("query: how do i pay my fee")
print(score("how do i pay my fee", df))

query: how do i pay my fee
                 question category  matched  confidence
0   how can i pay the fee  billing  fee pay         1.0
1  what is the annual fee  billing      fee         0.5


## Ques 3

Simple filter on the category column. Calling it with "account" since that is the
category my digit 7 entry landed in.

In [6]:
# Q3
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"]


print(same_category("account", df))

1                          how to reset password
4    how do i update my registered mobile number
Name: question, dtype: str


## Ques 4

Taking a keyword from the user, sticking it on the end of the first entry's
keywords, then writing the whole thing out as a csv.

In [7]:
# Q4
new_kw = input("new keyword for '" + df.loc[0, "question"] + "': ").strip()
if new_kw:
    df.loc[0, "keywords"] = df.loc[0, "keywords"] + " " + new_kw

fname = roll + "_faq_data.csv"
df.to_csv(fname, index=False)

print("keywords now:", df.loc[0, "keywords"])
print("saved to", fname)

keywords now: fee cost price charge
saved to 1024170378_faq_data.csv


## Ques 5

Count of entries in each category.

In [8]:
# Q5
print(df.groupby("category")["question"].count())

category
account    2
billing    2
general    2
Name: question, dtype: int64


## Ques 6

Same scoring as Q2, but instead of quietly taking the first row I check if more
than one entry sits at the top score and print all of them. "fee" is the tie case
since both billing entries have that keyword, and "reset password" is the clean
single match.

In [9]:
# Q6
def score_all(query, df):
    res = score(query, df)
    if res.empty:
        print("nothing matched:", query)
        return res
    top = res["confidence"].max()
    best = res[res["confidence"] == top]
    if len(best) > 1:
        print(len(best), "entries tied at", top)
    else:
        print("one best match at", top)
    print(best.to_string(index=False))
    return best


print("query: fee")
score_all("fee", df)

print()
print("query: reset password")
score_all("reset password", df)

query: fee
2 entries tied at 1.0
              question category matched  confidence
what is the annual fee  billing     fee         1.0
 how can i pay the fee  billing     fee         1.0

query: reset password
one best match at 1.0
             question category        matched  confidence
how to reset password  account password reset         1.0


,question,category,matched,confidence
0,how to reset password,account,password reset,1.0
